# Normalization Layers & Dropout from Scratch

**Difficulty**: 🟡 Medium

### Problem Statement

Normalization layers are in nearly every modern network: **BatchNorm** in ResNets and most CNNs, **LayerNorm** in every Transformer (ViT, GPT, the adaLN blocks of DiT), **GroupNorm** in diffusion U-Nets (Stable Diffusion's ResBlocks use `GroupNorm(32, C)`) and in detection/segmentation heads where per-GPU batches are tiny. All three compute the same thing,

$$\hat x = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad y = \gamma \odot \hat x + \beta,$$

and differ only in **which axes** $\mu$ and $\sigma^2$ are taken over. The details — biased vs unbiased variance, running statistics and their momentum, train vs eval behaviour, where $\epsilon$ goes — are exactly what interviewers probe and what silently breaks a model ported between frameworks. **Dropout** is the other layer whose behaviour flips between `model.train()` and `model.eval()`.

Implement all four as `nn.Module`s that match PyTorch exactly: outputs **and** gradients.

| layer | input | statistics over | affine $\gamma, \beta$ shape |
|---|---|---|---|
| `BatchNorm2d(C)` | `(N, C, H, W)` | $(N, H, W)$, one per channel | `(C,)` |
| `LayerNorm(shape)` | `(..., *shape)` | the trailing `shape` dims, one per sample/token | `shape` |
| `GroupNorm(G, C)` | `(N, C, *spatial)` | $(C/G, *\text{spatial})$, one per sample and group | `(C,)` |

---

### Requirements

1. **`BatchNorm2d(num_features, eps=1e-5, momentum=0.1)`** for inputs `(N, C, H, W)`.
   - Learnable `nn.Parameter`s `weight` ($\gamma$, init ones) and `bias` ($\beta$, init zeros), shape `(C,)`. Buffers `running_mean` (zeros) and `running_var` (ones), registered with `self.register_buffer`.
   - **Training** (`self.training == True`): per channel, $\mu_c$ = mean over $(N, H, W)$ and $\sigma_c^2$ = the **biased** variance (divide by $n = N \cdot H \cdot W$); normalize with them. Gradients must flow through $\mu$ and $\sigma^2$.
   - Then, without gradient, update the running stats with PyTorch's convention (momentum weights the **new** batch — the opposite of Keras):
     `running_mean = (1 - momentum) * running_mean + momentum * mean` and `running_var = (1 - momentum) * running_var + momentum * var_unbiased`, where $\sigma^2_{\text{unbiased}} = \sigma^2 \cdot \frac{n}{n-1}$.
   - **Eval**: normalize with `running_mean` / `running_var`, and don't update them.
2. **`LayerNorm(normalized_shape, eps=1e-5)`** — `normalized_shape` is an int or a tuple. Normalize each sample over its trailing `len(normalized_shape)` dims (biased variance). `weight` (ones) / `bias` (zeros) have shape `normalized_shape` (elementwise affine). Same behaviour in train and eval.
3. **`GroupNorm(num_groups, num_channels, eps=1e-5)`** — input `(N, C, *spatial)` with any number of spatial dims. Split the channels into `num_groups` groups of $C/G$ **consecutive** channels; normalize each (sample, group) over its channels and all spatial positions (biased variance). `weight` / `bias` are **per channel**, shape `(C,)`. Same in train and eval. ($G = 1$: LayerNorm over $(C, H, W)$ with per-channel affine; $G = C$: InstanceNorm.)
4. **`Dropout(p=0.5)`** — *inverted* dropout. Training: zero each element independently with probability `p` and scale the survivors by $1/(1-p)$, so $\mathbb{E}[\text{out}] = x$; draw a fresh mask on every call; don't modify the input in place. Eval: return the input unchanged. Edge cases: `p = 0` is the identity; `p = 1` returns all zeros (no NaN).
5. All four use `self.training` (set by `.train()` / `.eval()`), preserve the input's shape and dtype, keep $\epsilon$ **inside** the square root, use `self.eps`, and work after `.double()` (the grader compares in float64).

---

### Constraints

- ✅ Tensor ops: `mean`, `var(..., unbiased=False)` (or `correction=0`), `sqrt` / `rsqrt`, `reshape`, broadcasting, `torch.rand_like` / `torch.bernoulli`.
- ❌ No `F.batch_norm`, `F.layer_norm`, `F.group_norm`, `F.instance_norm`, `F.dropout` or their `nn.` module versions (the grader detects calls to them).
- ❌ Don't `.detach()` the batch statistics used for normalization — only the running-stat update is gradient-free.

---

<details>
  <summary>💡 Hint</summary>

  - For each layer, write down the reduction axes for an `(N, C, H, W)` input. Use `keepdim=True`, or view a `(C,)` vector as `(1, C, 1, 1)`, so broadcasting lines up.
  - `torch.var` is *unbiased* by default. Which variance makes $\hat x$ have exactly unit variance over the reduced axes?
  - The running-stat update is bookkeeping, not part of the model's differentiable function: what context manager makes that explicit?
  - GroupNorm: a single `reshape` of the input turns it into "normalize over the last dimension".
  - Dropout: which comparison between `torch.rand_like(x)` and `p` is true with probability $1 - p$? What happens to $x/(1-p)$ when $p = 1$?

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
torch.manual_seed(0)
feature_map = torch.randn(8, 6, 5, 5) * 2 + 1     # (N, C, H, W): a CNN activation
tokens = torch.randn(4, 10, 16) * 3 - 2           # (B, T, D): a transformer activation


def compare(mine, ref, x, name, seed=0):
    """Give both modules the same random affine params, then compare the output and the
    gradients w.r.t. input / weight / bias for a random upstream gradient, in float64."""
    mine, ref = mine.double(), ref.double()
    g = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for pname in ("weight", "bias"):
            q = getattr(ref, pname)
            q.copy_(torch.randn(q.shape, generator=g, dtype=torch.float64) * 0.5
                    + (1.0 if pname == "weight" else 0.0))
            getattr(mine, pname).copy_(q)
    x = x.double()
    gy = torch.randn(x.shape, generator=g, dtype=torch.float64)   # NOT ones: d sum(BN(x))/dx ~ 0
    results = []
    for m in (mine, ref):
        m.weight.grad = m.bias.grad = None
        xx = x.clone().requires_grad_()
        y = m(xx)
        y.backward(gy)
        results.append((y.detach(), xx.grad, m.weight.grad, m.bias.grad))
    for what, a, b in zip(("output", "d/dx", "d/dweight", "d/dbias"), *results):
        assert a is not None, f"{name}: {what} is None"
        assert torch.allclose(a, b, rtol=1e-6, atol=1e-8), \
            f"{name}: {what} mismatch (max diff {(a - b).abs().max().item():.2e})"
    print(f"{name}: output and gradients match PyTorch")

In [ ]:
class BatchNorm2d(nn.Module):
    """Batch normalization for (N, C, H, W) inputs, matching nn.BatchNorm2d."""

    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        super().__init__()
        # TODO: self.weight / self.bias as nn.Parameter of shape (C,) (ones / zeros);
        #       running_mean (zeros) / running_var (ones) via self.register_buffer
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W)
        Returns:
            (N, C, H, W). Training: normalize with the batch mean / biased variance over
            (N, H, W) and update the running stats (unbiased variance) under no_grad.
            Eval: normalize with the running stats.
        """
        # TODO
        raise NotImplementedError

In [ ]:
class LayerNorm(nn.Module):
    """Layer normalization over the trailing `normalized_shape` dims, matching nn.LayerNorm."""

    def __init__(self, normalized_shape, eps: float = 1e-5):
        super().__init__()
        # TODO: accept an int or a tuple; weight (ones) / bias (zeros) of shape normalized_shape
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (..., *normalized_shape)
        Returns:
            same shape, each sample normalized over its last len(normalized_shape) dims.
        """
        # TODO
        raise NotImplementedError

In [ ]:
class GroupNorm(nn.Module):
    """Group normalization for (N, C, *spatial) inputs, matching nn.GroupNorm."""

    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        # TODO: weight (ones) / bias (zeros) of shape (num_channels,)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, *spatial), C divisible by num_groups
        Returns:
            same shape; each sample's group of C // G consecutive channels is normalized over
            (C // G, *spatial), then the per-channel affine is applied.
        """
        # TODO
        raise NotImplementedError

In [ ]:
class Dropout(nn.Module):
    """Inverted dropout, matching nn.Dropout."""

    def __init__(self, p: float = 0.5):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: any shape
        Returns:
            training: x * mask / (1 - p) with P(mask = 1) = 1 - p, fresh mask each call
                      (p = 0 -> x, p = 1 -> zeros, never NaN)
            eval:     x unchanged
        """
        # TODO
        raise NotImplementedError

In [ ]:
# Test 1 — BatchNorm2d in training mode: output and all gradients
for shape in [(8, 6, 5, 5), (2, 3, 1, 1)]:
    compare(BatchNorm2d(shape[1]), nn.BatchNorm2d(shape[1]), torch.randn(shape) * 2 + 1,
            f"BatchNorm2d train {shape}")

# running stats after several batches of different shapes, then eval mode
mine, ref = BatchNorm2d(6, momentum=0.3).double(), nn.BatchNorm2d(6, momentum=0.3).double()
for step in range(5):
    xb = torch.randn(2 + step, 6, 4, 3 + step, dtype=torch.float64) * (1 + step) + step
    mine(xb), ref(xb)
assert torch.allclose(mine.running_mean, ref.running_mean), "running_mean differs from PyTorch"
assert torch.allclose(mine.running_var, ref.running_var), "running_var differs (unbiased variance? momentum convention?)"
print("running stats match after 5 steps")
mine.eval(), ref.eval()
compare(mine, ref, feature_map, "BatchNorm2d eval (running stats)")
assert "running_mean" in dict(mine.named_buffers()), "use register_buffer for running stats"

In [ ]:
# Test 2 — LayerNorm and GroupNorm
compare(LayerNorm(16), nn.LayerNorm(16), tokens, "LayerNorm(16) on (B, T, D)")
compare(LayerNorm((10, 16)), nn.LayerNorm((10, 16)), tokens, "LayerNorm((10, 16))")
compare(LayerNorm((6, 5, 5)), nn.LayerNorm((6, 5, 5)), feature_map, "LayerNorm((C, H, W))")
for G in (1, 2, 3, 6):
    compare(GroupNorm(G, 6), nn.GroupNorm(G, 6), feature_map, f"GroupNorm({G}, 6)")
compare(GroupNorm(2, 4), nn.GroupNorm(2, 4), torch.randn(3, 4, 7), "GroupNorm(2, 4) on (N, C, L)")
# tiny variance: eps placement matters
compare(LayerNorm(16, eps=1e-3), nn.LayerNorm(16, eps=1e-3), tokens * 1e-3, "LayerNorm, tiny-variance input")

In [ ]:
# Test 3 — Dropout
torch.manual_seed(0)
p = 0.3
drop = Dropout(p)
x = torch.randn(400, 500)
out = drop(x)                                       # modules start in training mode
kept = out != 0
print(f"keep rate {kept.float().mean().item():.4f} (expected {1 - p})")
assert abs(kept.float().mean().item() - (1 - p)) < 0.01
assert torch.allclose(out[kept], x[kept] / (1 - p)), "survivors must be scaled by 1 / (1 - p)"
assert abs(drop(torch.ones(400, 500)).mean().item() - 1.0) < 0.02, "E[dropout(x)] must equal x"
assert not torch.equal(drop(x) != 0, kept), "each call must draw a fresh mask"
drop.eval()
assert torch.equal(drop(x), x), "eval mode must be the identity"
assert torch.equal(Dropout(0.0)(x), x), "p = 0 must be the identity"
z = Dropout(1.0)(x)
assert torch.equal(z, torch.zeros_like(x)) and not z.isnan().any(), "p = 1 must give zeros, not NaN"
print("Dropout OK")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("normalization-layers", BatchNorm2d=BatchNorm2d, LayerNorm=LayerNorm, GroupNorm=GroupNorm, Dropout=Dropout)